In [1]:
import cv2
import os
import glob
import numpy as np

save_dir = "data/raw/blue_kyanite"
os.makedirs(save_dir, exist_ok=True)

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

saved_files = sorted(glob.glob(os.path.join(save_dir, "cm_*.png")))
img_counter = len(saved_files)

window_title = "SPEC-0002 Capture (Targeting Rich Mineral Zones)"
cv2.namedWindow(window_title, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_title, 960, 540)

print("Look for rich color contacts")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        continue

    # Compute color richness (Mean Saturation in HSV space)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1]
    color_score = float(np.mean(sat))

    display_frame = frame.copy()
    
    # Visual indicator: Green when framing vivid mineral zones, Yellow when plain
    status_color = (0, 255, 0) if color_score > 35 else (0, 200, 255)
    overlay_text = f"Saved: {img_counter} | Color Score: {color_score:.1f}"
    cv2.putText(display_frame, overlay_text, (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, status_color, 2, cv2.LINE_AA)
    
    if color_score <= 35:
        cv2.putText(display_frame, "Plain matrix area - pan to color zone", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2, cv2.LINE_AA)

    cv2.imshow(window_title, display_frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q') or key == 27:
        break
    elif key == 32:  # SPACE
        img_counter += 1
        filepath = os.path.join(save_dir, f"cm_{img_counter:03d}.png")
        cv2.imwrite(filepath, frame)
        saved_files.append(filepath)
        print(f"Captured: {filepath} (Color Score: {color_score:.1f})")
    elif key in (ord('d'), ord('D')):
        if saved_files:
            last = saved_files.pop()
            if os.path.exists(last):
                os.remove(last)
                img_counter -= 1
                print(f"Deleted: {last}")

cap.release()
cv2.destroyAllWindows()
print(f"Collection complete: {len(saved_files)} color-rich frames saved.")

Look for rich color contacts
Captured: data/raw/blue_kyanite\cm_001.png (Color Score: 64.8)
Captured: data/raw/blue_kyanite\cm_002.png (Color Score: 57.0)
Captured: data/raw/blue_kyanite\cm_003.png (Color Score: 78.6)
Captured: data/raw/blue_kyanite\cm_004.png (Color Score: 57.6)
Captured: data/raw/blue_kyanite\cm_005.png (Color Score: 51.2)
Captured: data/raw/blue_kyanite\cm_006.png (Color Score: 54.5)
Captured: data/raw/blue_kyanite\cm_007.png (Color Score: 60.3)
Captured: data/raw/blue_kyanite\cm_008.png (Color Score: 68.9)
Captured: data/raw/blue_kyanite\cm_009.png (Color Score: 90.7)
Captured: data/raw/blue_kyanite\cm_010.png (Color Score: 66.7)
Captured: data/raw/blue_kyanite\cm_011.png (Color Score: 76.2)
Captured: data/raw/blue_kyanite\cm_012.png (Color Score: 78.1)
Captured: data/raw/blue_kyanite\cm_013.png (Color Score: 100.1)
Captured: data/raw/blue_kyanite\cm_014.png (Color Score: 93.6)
Captured: data/raw/blue_kyanite\cm_015.png (Color Score: 81.7)
Captured: data/raw/blue_k

In [2]:
import os
import glob
import cv2
import pandas as pd
import matplotlib.pyplot as plt

# Directory containing your SPEC-0003 captures
raw_dir = "data/raw/blue_kyanite"
image_paths = sorted(glob.glob(os.path.join(raw_dir, "*.png")))

focus_records = []
for p in image_paths:
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    # Compute Laplacian variance (second derivative focus metric)
    score = cv2.Laplacian(img, cv2.CV_64F).var()
    focus_records.append({
        "filepath": p,
        "filename": os.path.basename(p),
        "focus_score": round(score, 2)
    })

df_focus = pd.DataFrame(focus_records).sort_values(by="focus_score", ascending=False).reset_index(drop=True)

# Visualize top 4 vs. bottom 4 focus scores
print("Focus Quality Distribution:")
print(f"  Maximum Sharpness: {df_focus['focus_score'].max():.1f}")
print(f"  Median Sharpness:  {df_focus['focus_score'].median():.1f}")
print(f"  Minimum Sharpness: {df_focus['focus_score'].min():.1f}")

top_12_filenames = set(df_focus.head(12)["filename"])
print(f"\nTop 12 Sharpest Frames Selected:\n{list(top_12_filenames)}")

Focus Quality Distribution:
  Maximum Sharpness: 83.5
  Median Sharpness:  45.7
  Minimum Sharpness: 21.0

Top 12 Sharpest Frames Selected:
['cm_023.png', 'cm_017.png', 'cm_021.png', 'cm_022.png', 'cm_030.png', 'cm_020.png', 'cm_029.png', 'cm_028.png', 'cm_019.png', 'cm_003.png', 'cm_016.png', 'cm_026.png']


In [3]:
import numpy as np
from skimage.feature import graycomatrix, graycoprops
from sklearn.cluster import KMeans

def extract_kyanite_metrics(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    
    h, w, _ = img.shape
    crop = img[40:h-40, 80:w-80]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    
    # 1. Gradient Orientation Vector Field (Sobel)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    ang = ang % 180.0  # Fold orientation into [0, 180)
    
    # Filter for significant structural edges
    mag_mask = mag > np.percentile(mag, 75)
    valid_angles = ang[mag_mask]
    hist, bin_edges = np.histogram(valid_angles, bins=36, range=(0, 180))
    dominant_angle = float(bin_edges[np.argmax(hist)] + 2.5)
    alignment_strength = float(np.max(hist) / (np.mean(hist) + 1e-5))
    
    # 2. GLCM Texture Dynamics
    center_patch = gray[gray.shape[0]//4:3*gray.shape[0]//4, gray.shape[1]//4:3*gray.shape[1]//4]
    glcm = graycomatrix(center_patch, distances=[1], angles=[0, np.pi/2], levels=256, symmetric=True, normed=True)
    contrast = float(graycoprops(glcm, 'contrast').mean())
    homogeneity = float(graycoprops(glcm, 'homogeneity').mean())
    
    # 3. 3-Phase Color Segmentation (CIE-L*a*b*)
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
    # Balanced weights across lightness and chromatic channels
    weights = np.array([0.5, 2.0, 2.0], dtype=np.float32)
    weighted_px = (lab * weights).reshape(-1, 3)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = kmeans.fit_predict(weighted_px)
    counts = np.bincount(labels, minlength=3)
    pcts = (counts / counts.sum()) * 100.0
    
    # Sort clusters by lightness
    cluster_l = [np.mean(crop_rgb.reshape(-1, 3)[labels == i]) for i in range(3)]
    sorted_indices = np.argsort(cluster_l)  # 0: Dark matrix, 1: Intermediate blade, 2: Bright facet
    
    fname = os.path.basename(img_path)
    return {
        "filename": fname,
        "is_top_12": fname in top_12_filenames,
        "dominant_angle_deg": round(dominant_angle, 1),
        "alignment_strength": round(alignment_strength, 2),
        "glcm_contrast": round(contrast, 2),
        "glcm_homogeneity": round(homogeneity, 3),
        "pct_dark_matrix": round(float(pcts[sorted_indices[0]]), 2),
        "pct_intermediate_blade": round(float(pcts[sorted_indices[1]]), 2),
        "pct_bright_facet": round(float(pcts[sorted_indices[2]]), 2)
    }

# Run extraction across all frames
results = [extract_kyanite_metrics(p) for p in image_paths if extract_kyanite_metrics(p)]
df_all = pd.DataFrame(results)

# Compare Metrics: All 31 vs. Top 12
summary_comparison = pd.DataFrame({
    "Metric": ["GLCM Contrast", "GLCM Homogeneity", "Alignment Strength", "Dominant Angle (°)", "Bright Facet %"],
    "All 31 Frames (Mean ± Std)": [
        f"{df_all['glcm_contrast'].mean():.2f} ± {df_all['glcm_contrast'].std():.2f}",
        f"{df_all['glcm_homogeneity'].mean():.3f} ± {df_all['glcm_homogeneity'].std():.3f}",
        f"{df_all['alignment_strength'].mean():.2f} ± {df_all['alignment_strength'].std():.2f}",
        f"{df_all['dominant_angle_deg'].mean():.1f}° ± {df_all['dominant_angle_deg'].std():.1f}°",
        f"{df_all['pct_bright_facet'].mean():.2f}% ± {df_all['pct_bright_facet'].std():.2f}%"
    ],
    "Top 12 Sharpest (Mean ± Std)": [
        f"{df_all[df_all['is_top_12']]['glcm_contrast'].mean():.2f} ± {df_all[df_all['is_top_12']]['glcm_contrast'].std():.2f}",
        f"{df_all[df_all['is_top_12']]['glcm_homogeneity'].mean():.3f} ± {df_all[df_all['is_top_12']]['glcm_homogeneity'].std():.3f}",
        f"{df_all[df_all['is_top_12']]['alignment_strength'].mean():.2f} ± {df_all[df_all['is_top_12']]['alignment_strength'].std():.2f}",
        f"{df_all[df_all['is_top_12']]['dominant_angle_deg'].mean():.1f}° ± {df_all[df_all['is_top_12']]['dominant_angle_deg'].std():.1f}°",
        f"{df_all[df_all['is_top_12']]['pct_bright_facet'].mean():.2f}% ± {df_all[df_all['is_top_12']]['pct_bright_facet'].std():.2f}%"
    ]
})

print("\n--- Comparative Analysis Table ---")
print(summary_comparison.to_string(index=False))


--- Comparative Analysis Table ---
            Metric All 31 Frames (Mean ± Std) Top 12 Sharpest (Mean ± Std)
     GLCM Contrast              21.40 ± 11.66                30.31 ± 12.96
  GLCM Homogeneity              0.361 ± 0.060                0.325 ± 0.047
Alignment Strength                1.82 ± 0.35                  1.99 ± 0.42
Dominant Angle (°)              92.5° ± 20.1°                 92.5° ± 0.0°
    Bright Facet %             38.87% ± 8.69%               36.44% ± 6.83%


## Specimen SPEC-0003: Bladed Blue Kyanite

### Focus Filtering vs. Macro Population Diagnostics
* **Cleavage Axis Locking:** Restricting analysis to the top 12 frames via Laplacian variance filtering eliminates off-axis fracture noise, resulting in absolute angular convergence at **92.5° ± 0.0°** (compared to 92.5° ± 20.1° across the full 31 frames).
* **High-Frequency Step Recovery:** Filtering out focal depth falloff elevated mean GLCM contrast from **21.40 to 30.31**, accurately reflecting the stepped knife-edge topography of the crystal blades.
* **Phase Stability:** Surface facet coverage remained consistent across both subsets (~36.4% to 38.9%), demonstrating that phase ratio quantification is robust to slight focal plane variations.

In [4]:
import json

# 1. Filter dataset to top 12 sharpest
df_kyanite_top12 = df_all[df_all["is_top_12"]].copy()
df_kyanite_top12["specimen_id"] = "SPEC-0003"
df_kyanite_top12["mineral_name"] = "Bladed Blue Kyanite"

# 2. Append to docs/data.json
with open("docs/data.json", "r") as f:
    master_data = json.load(f)

# Avoid duplicate entries if re-run
master_data = [row for row in master_data if row.get("specimen_id") != "SPEC-0003"]
master_data.extend(df_kyanite_top12.to_dict(orient="records"))

with open("docs/data.json", "w") as f:
    json.dump(master_data, f, indent=2)

# 3. Create thumbnail from the sharpest frame (#1 ranked)
best_frame_path = df_focus.iloc[0]["filepath"]
img_best = cv2.imread(best_frame_path)
h, w = img_best.shape[:2]
scale = 480.0 / w
thumb = cv2.resize(img_best, (480, int(h * scale)), interpolation=cv2.INTER_AREA)

cv2.putText(thumb, "SPEC-0003 / BLADED BLUE KYANITE", (15, int(h * scale) - 15),
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
cv2.imwrite("docs/assets/thumbnails/specimen_0003_thumb.jpg", thumb, [cv2.IMWRITE_JPEG_QUALITY, 80])

print(f"Appended {len(df_kyanite_top12)} records to docs/data.json.")
print("Thumbnail saved to docs/assets/thumbnails/specimen_0003_thumb.jpg")

Appended 12 records to docs/data.json.
Thumbnail saved to docs/assets/thumbnails/specimen_0003_thumb.jpg
